# Data Load

In [1]:
import os
import requests
import tarfile

URL = "https://s3.amazonaws.com/content.udacity-data.com/nd089/flower_data.tar.gz"
DATA_DIR = "data"
TAR_PATH = os.path.join(DATA_DIR, "flower_data.tar.gz")
EXTRACT_DIR = os.path.join(DATA_DIR, "flowers")

os.makedirs(DATA_DIR, exist_ok=True)

# Download
if not os.path.exists(TAR_PATH):
    print("Downloading dataset...")
    response = requests.get(URL, stream=True)
    response.raise_for_status()
    with open(TAR_PATH, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    print("Download complete.")
else:
    print("Tar file already exists, skipping download.")

# Extract
if not os.path.exists(EXTRACT_DIR):
    print("Extracting dataset...")
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with tarfile.open(TAR_PATH, mode="r:gz") as tar:
        tar.extractall(path=EXTRACT_DIR)
    print("Extraction complete.")
else:
    print("Dataset already extracted.")



Download complete.
Extracting dataset...


/tmp/ipython-input-1154811326.py:30: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=EXTRACT_DIR)


Extraction complete.


In [2]:
# Install the necessary library (usually already present, but safe to include)
# !pip install torch torchvision

# 3. Verify the files are set up
!ls -F
!ls data/flowers

data/  sample_data/
test  train  valid


In [3]:
# Download json file from my Google cloud

json_url = "https://storage.googleapis.com/data_files_soliman/udacity/Python_AI/cat_to_name.json"

!wget -O cat_to_name.json {json_url}

--2025-12-14 15:37:35--  https://storage.googleapis.com/data_files_soliman/udacity/Python_AI/cat_to_name.json
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.68.207, 172.253.118.207, 74.125.24.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.68.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2218 (2.2K) [application/json]
Saving to: ‘cat_to_name.json’

cat_to_name.json    100%[===================>]   2.17K  --.-KB/s    in 0s      

2025-12-14 15:37:37 (30.1 MB/s) - ‘cat_to_name.json’ saved [2218/2218]



# Train file

In [4]:
%%writefile train.py
# ==========================================================================================
# train.py (Pasted from previous response)
# ==========================================================================================
import torch
from torch import nn
from torch import optim
from torchvision import datasets, transforms, models
from collections import OrderedDict
import argparse
import os

def get_input_args():
    parser = argparse.ArgumentParser(description='Train a new image classifier network.')
    parser.add_argument('data_dir', type=str,
                        help='Path to the dataset directory (e.g., flowers)')
    parser.add_argument('--save_dir', type=str, default='.',
                        help='Directory to save checkpoints')
    parser.add_argument('--arch', type=str, default='vgg16',
                        choices=['vgg16', 'densenet121'],
                        help='Pre-trained model architecture to use (vgg16 or densenet121)')
    parser.add_argument('--learning_rate', type=float, default=0.001,
                        help='Model learning rate')
    parser.add_argument('--hidden_units', type=int, default=4096,
                        help='Number of hidden units in the classifier')
    parser.add_argument('--epochs', type=int, default=3,
                        help='Number of training epochs')
    parser.add_argument('--gpu', action='store_true',
                        help='Use GPU for training if available')
    return parser.parse_args()

def get_data_loaders(data_dir):
    train_dir = os.path.join(data_dir, 'train')
    valid_dir = os.path.join(data_dir, 'valid')
    test_dir = os.path.join(data_dir, 'test')

    train_transforms = transforms.Compose([
        transforms.RandomRotation(30),
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    valid_test_transforms = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    train_data = datasets.ImageFolder(train_dir, transform=train_transforms)
    valid_data = datasets.ImageFolder(valid_dir, transform=valid_test_transforms)
    test_data = datasets.ImageFolder(test_dir, transform=valid_test_transforms)

    trainloader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True)
    validloader = torch.utils.data.DataLoader(valid_data, batch_size=64)
    testloader = torch.utils.data.DataLoader(test_data, batch_size=64)

    return trainloader, validloader, testloader, train_data.class_to_idx

def build_model(arch, hidden_units):
    if arch == 'vgg16':
        model = models.vgg16(pretrained=True)
        input_size = model.classifier[0].in_features
    elif arch == 'densenet121':
        model = models.densenet121(pretrained=True)
        input_size = model.classifier.in_features
    else:
        raise ValueError(f"Architecture '{arch}' not supported. Choose 'vgg16' or 'densenet121'.")

    for param in model.parameters():
        param.requires_grad = False

    output_size = 102

    classifier = nn.Sequential(OrderedDict([
        ('fc1', nn.Linear(input_size, hidden_units)),
        ('relu1', nn.ReLU()),
        ('dropout1', nn.Dropout(0.2)),
        ('fc2', nn.Linear(hidden_units, output_size)),
        ('output', nn.LogSoftmax(dim=1))
    ]))

    if arch == 'vgg16':
        model.classifier = classifier
    elif arch == 'densenet121':
        model.classifier = classifier

    return model, input_size

def validate_model(model, criterion, validloader, device):
    valid_loss = 0
    accuracy = 0
    model.eval()

    with torch.no_grad():
        for inputs, labels in validloader:
            inputs, labels = inputs.to(device), labels.to(device)
            logps = model.forward(inputs)
            batch_loss = criterion(logps, labels)

            valid_loss += batch_loss.item()

            ps = torch.exp(logps)
            top_p, top_class = ps.topk(1, dim=1)
            equals = top_class == labels.view(*top_class.shape)
            accuracy += torch.mean(equals.type(torch.FloatTensor)).item()

    return valid_loss/len(validloader), accuracy/len(validloader)

def train_model(model, trainloader, validloader, criterion, optimizer, epochs, device):
    print_every = 40
    steps = 0

    model.to(device)

    for epoch in range(epochs):
        running_loss = 0
        model.train()
        for inputs, labels in trainloader:
            steps += 1
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            logps = model.forward(inputs)
            loss = criterion(logps, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            if steps % print_every == 0:
                valid_loss, valid_accuracy = validate_model(model, criterion, validloader, device)

                print(f"Epoch {epoch+1}/{epochs}.. "
                      f"Train Loss: {running_loss/print_every:.3f}.. "
                      f"Validation Loss: {valid_loss:.3f}.. "
                      f"Validation Accuracy: {valid_accuracy:.3f}")

                running_loss = 0
                model.train()

    print("Training complete.")


def save_checkpoint(model, input_size, arch, class_to_idx, save_dir):
    os.makedirs(save_dir, exist_ok=True)

    checkpoint = {
        'input_size': input_size,
        'output_size': 102,
        'structure': arch,
        'classifier': model.classifier,
        'state_dict': model.state_dict(),
        'class_to_idx': class_to_idx
    }

    checkpoint_path = os.path.join(save_dir, 'checkpoint.pth')
    # Save with weights_only=False for compatibility when loading in the future
    torch.save(checkpoint, checkpoint_path)
    print(f"Model checkpoint saved to {checkpoint_path}")


def main():
    args = get_input_args()

    device = torch.device("cuda" if args.gpu and torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    trainloader, validloader, testloader, class_to_idx = get_data_loaders(args.data_dir)
    print("Data loading complete.")

    model, input_size = build_model(args.arch, args.hidden_units)

    criterion = nn.NLLLoss()
    optimizer = optim.Adam(model.classifier.parameters(), lr=args.learning_rate)

    print(f"Starting training with architecture: {args.arch}, LR: {args.learning_rate}, Epochs: {args.epochs}, Hidden Units: {args.hidden_units}")

    train_model(model, trainloader, validloader, criterion, optimizer, args.epochs, device)

    model.class_to_idx = class_to_idx
    save_checkpoint(model, input_size, args.arch, class_to_idx, args.save_dir)


if __name__ == '__main__':
    main()

Writing train.py


## Terminal run for training

In [5]:
!python train.py data/flowers --arch vgg16 --learning_rate 0.001 --hidden_units 512 --epochs 3 --gpu

Using device: cuda
Data loading complete.
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100% 528M/528M [00:05<00:00, 93.5MB/s]
Starting training with architecture: vgg16, LR: 0.001, Epochs: 3, Hidden Units: 512
Epoch 1/3.. Train Loss: 3.317.. Validation Loss: 1.606.. Validation Accuracy: 0.631
Epoch 1

In [6]:
# !python train.py data/flowers --arch vgg16 --learning_rate 0.01 --hidden_units 512 --epochs 5 --gpu --save_dir saved_models/

# Predict file

In [7]:
%%writefile predict.py
# ==========================================================================================
# predict.py (Pasted from previous response)
# ==========================================================================================

import torch
from torch import nn
from torchvision import models
from PIL import Image
import numpy as np
import json
import argparse
import os
from collections import OrderedDict

def get_input_args():
    parser = argparse.ArgumentParser(description='Predict flower class from an image.')
    parser.add_argument('image_path', type=str,
                        help='Path to the image file')
    parser.add_argument('checkpoint', type=str,
                        help='Path to the model checkpoint file')
    parser.add_argument('--top_k', type=int, default=1,
                        help='Return the top K most likely classes')
    parser.add_argument('--category_names', type=str, default='cat_to_name.json',
                        help='Path to a JSON file mapping categories to real names')
    parser.add_argument('--gpu', action='store_true',
                        help='Use GPU for inference if available')
    return parser.parse_args()


def load_checkpoint(filepath):
    """
    Loads a checkpoint file and rebuilds the model.
    """
    # Use weights_only=False to load the custom Sequential object safely
    checkpoint = torch.load(filepath, weights_only=False)

    arch = checkpoint['structure']

    if arch == 'vgg16':
        model = models.vgg16(pretrained=True)
    elif arch == 'densenet121':
        model = models.densenet121(pretrained=True)
    else:
        raise ValueError(f"Unsupported architecture: {arch}")

    # Freeze parameters
    for param in model.parameters():
        param.requires_grad = False

    # Load the classifier and state dictionary
    model.classifier = checkpoint['classifier']
    model.load_state_dict(checkpoint['state_dict'])
    model.class_to_idx = checkpoint['class_to_idx']

    return model, arch


def process_image(image_path):
    ''' Scales, crops, and normalizes a PIL image for a PyTorch model,
        returns an Numpy array.
    '''
    img = Image.open(image_path)

    # 1. Resize: shortest side to 256
    if img.size[0] > img.size[1]:   # landscape
        img.thumbnail((10000, 256))
    else:                           # portrait
        img.thumbnail((256, 10000))

    # 2. Crop: center 224x224
    left = (img.width - 224) / 2
    bottom = (img.height - 224) / 2
    right = left + 224
    top = bottom + 224
    img = img.crop((left, bottom, right, top))

    # 3. Normalize: Convert to 0-1 float and normalize
    np_img = np.array(img) / 255
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    np_img = (np_img - mean) / std

    # 4. Transpose: PyTorch expects color channel first
    np_img = np_img.transpose((2, 0, 1))

    return np_img


def predict(image_path, model, topk, device):
    ''' Predict the class (or classes) of an image using a trained deep learning model.
    '''

    model.to(device)
    model.eval()

    # Pre-process image
    img = process_image(image_path)

    # Convert numpy to torch tensor and add batch dimension
    img_tensor = torch.from_numpy(img).type(torch.FloatTensor)
    img_tensor = img_tensor.unsqueeze(0)
    img_tensor = img_tensor.to(device)

    # Inference (Forward pass)
    with torch.no_grad():
        logps = model.forward(img_tensor)
        ps = torch.exp(logps)
        top_p, top_class = ps.topk(topk, dim=1)

    # Convert results to standard Python lists
    top_p = top_p.cpu().numpy()[0].tolist()
    top_class = top_class.cpu().numpy()[0].tolist()

    # Map indices back to class labels
    idx_to_class = {v: k for k, v in model.class_to_idx.items()}
    top_classes = [idx_to_class[c] for c in top_class]

    return top_p, top_classes


def main():
    args = get_input_args()

    device = torch.device("cuda" if args.gpu and torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Load model
    model, arch = load_checkpoint(args.checkpoint)
    print(f"Model ({arch}) loaded from {args.checkpoint}.")

    # Load category names
    with open(args.category_names, 'r') as f:
        cat_to_name = json.load(f)
    print(f"Category names loaded from {args.category_names}.")

    # Predict
    probs, classes = predict(args.image_path, model, args.top_k, device)

    # Display results
    print(f"\n--- Prediction Results for {args.image_path} ---")

    if args.top_k == 1:
        flower_name = cat_to_name.get(classes[0], classes[0])
        print(f"The most likely flower is: {flower_name}")
        print(f"Probability: {probs[0]*100:.2f}%")
    else:
        # Create a list of (name, probability) tuples
        results = []
        for class_index, prob in zip(classes, probs):
            flower_name = cat_to_name.get(class_index, class_index)
            results.append((flower_name, prob))

        print(f"Top {args.top_k} Predictions:")
        for name, prob in results:
            print(f"  {name: <25}: {prob*100:.2f}%")

    print("-" * 40)


if __name__ == '__main__':
    main()

Writing predict.py


## Terminal run for predication

In [8]:
!python predict.py data/flowers/test/1/image_06743.jpg checkpoint.pth

Using device: cpu
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Model (vgg16) loaded from checkpoint.pth.
Category names loaded from cat_to_name.json.

--- Prediction Results for data/flowers/test/1/image_06743.jpg ---
The most likely flower is: pink primrose
Probability: 99.68%
----------------------------------------


In [9]:
!python predict.py data/flowers/test/43/image_02369.jpg checkpoint.pth --top_k 10 --gpu

Using device: cuda
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Model (vgg16) loaded from checkpoint.pth.
Category names loaded from cat_to_name.json.

--- Prediction Results for data/flowers/test/43/image_02369.jpg ---
Top 10 Predictions:
  sword lily               : 90.55%
  canna lily               : 3.70%
  carnation                : 1.44%
  hibiscus                 : 0.91%
  daffodil                 : 0.73%
  watercress